In [ ]:
from google.colab import userdata
from pathlib import Path
import os
import subprocess
import tempfile

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
PROJECT_DIR = Path("/content/project")

token = userdata.get("GITHUB_TOKEN")
if not token:
    raise ValueError("Chưa thiết lập GITHUB_TOKEN trong Colab Secrets")

with tempfile.TemporaryDirectory() as tmp:
    askpass = Path(tmp) / "askpass.sh"
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *Password*) printf "%s\\n" "$COLAB_GITHUB_TOKEN" ;;\n'
        'esac\n'
    )
    askpass.chmod(0o700)

    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["COLAB_GITHUB_TOKEN"] = token

    if (PROJECT_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "-C", str(PROJECT_DIR), "checkout", "main"],
            check=True,
            env=env,
        )
        subprocess.run(
            ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", "main"],
            check=True,
            env=env,
        )
    else:
        subprocess.run(
            ["git", "clone", "--branch", "main", REPO_URL, str(PROJECT_DIR)],
            check=True,
            env=env,
        )

del token

print("Repository ready:", PROJECT_DIR)

In [ ]:
%cd /content/project

%pip install -q -e ".[data]"

from vietnamese_topic_classification.config import load_domain_config

config = load_domain_config()

print("Task:", config["task"])
print("Classes:", config["num_classes"])
print("Dataset:", config["dataset"]["id"])
print("Revision:", config["dataset"]["revision"])

In [ ]:
from google.colab import drive
from pathlib import Path

MOUNT = "/content/vtsnlp_drive"

drive.mount(MOUNT)

DRIVE_ROOT = (
    Path(MOUNT)
    / "MyDrive"
    / "vietnamese-multilabel-topic-classification"
)

if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy project folder: {DRIVE_ROOT}"
    )

print("Google Drive connected!")
print("Project:", DRIVE_ROOT)

In [ ]:
import subprocess
import sys

SAMPLE_SIZE = 50_000
SEED = 42
SHUFFLE_BUFFER = 10_000

revision = config["dataset"]["revision"]

STAMP = (
    f"{revision[:10]}_seed{SEED}"
    f"_buf{SHUFFLE_BUFFER}_n{SAMPLE_SIZE}"
)

BRONZE_FILE = (
    DRIVE_ROOT / "01_bronze"
    / f"vtsnlp_raw_sample_{STAMP}.jsonl"
)

MANIFEST_FILE = (
    DRIVE_ROOT / "00_source_metadata"
    / f"source_manifest_{STAMP}.json"
)

REPORT_FILE = (
    DRIVE_ROOT / "05_reports"
    / f"bronze_ingestion_{STAMP}.json"
)

existing = [
    p for p in (BRONZE_FILE, MANIFEST_FILE, REPORT_FILE)
    if p.exists()
]

if existing:
    raise FileExistsError(
        f"Snapshot đã tồn tại: {existing}. "
        "Không chạy ghi đè dữ liệu."
    )

command = [
    sys.executable,
    str(PROJECT_DIR / "scripts/01_ingest_bronze.py"),
    "--output-root", str(DRIVE_ROOT),
    "--sample-size", str(SAMPLE_SIZE),
    "--seed", str(SEED),
    "--shuffle-buffer", str(SHUFFLE_BUFFER),
]

subprocess.run(
    command,
    cwd=str(PROJECT_DIR),
    check=True,
)

print("Bronze ingestion completed!")

In [ ]:
import hashlib
import json

assert BRONZE_FILE.is_file(), "Missing Bronze file"
assert MANIFEST_FILE.is_file(), "Missing manifest"
assert REPORT_FILE.is_file(), "Missing report"

manifest = json.loads(
    MANIFEST_FILE.read_text(encoding="utf-8")
)

report = json.loads(
    REPORT_FILE.read_text(encoding="utf-8")
)

digest = hashlib.sha256()
rows = 0

with BRONZE_FILE.open("rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        digest.update(chunk)
        rows += chunk.count(b"\n")

assert rows == manifest["sample_size"]
assert digest.hexdigest() == manifest["sha256"]

print("BRONZE VALIDATION PASSED")
print("Records:", rows)
print("File size:", BRONZE_FILE.stat().st_size)
print("SHA256:", digest.hexdigest())
print("Domains:", len(report["domain_counts"]))
print("Source:", manifest["dataset"])
print("Revision:", manifest["revision"])